# MvDEC REUTERS seed-42 paper-figure audit

Evaluation-only diagnostic. This notebook reuses one complete trained run, performs no training, and reports Equation 4 plus both conflicting Figure 1/Figure 2 full-output interpretations. None of the three is labeled exact paper truth.


In [ ]:
import hashlib
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

import numpy as np
from sklearn import __version__ as sklearn_version

AUDIT_COMMIT = "059ef8ca9e0c5594bc1c39ccf50063163af29c81"
SOURCE_DATASET = "vandatzz/mvdec-reuters-v2-seed42-audit-source"
SOURCE_PROTOCOL = "mvdec_2025_legacy_scaled_centered_alignment_100_full_batch_v1"
SOURCE_TRAINING_COMMIT = "499e4668264203ca24f70cdea6a3a2f132e86b90"
EXPECTED_NUMPY = "2.1.3"
EXPECTED_SKLEARN = "1.6.1"
HARD_TIMEOUT_SECONDS = 20 * 60
INPUT_ROOT = Path("/kaggle/input")
WORKING_ROOT = Path("/kaggle/working")
REPO = WORKING_ROOT / "mvdec_fpmax"
OUTPUT_DIR = WORKING_ROOT / "mvdec-reuters-paper-figure-audit-seed42"

assert np.__version__ == EXPECTED_NUMPY, np.__version__
assert sklearn_version == EXPECTED_SKLEARN, sklearn_version
print("NumPy", np.__version__)
print("scikit-learn", sklearn_version)


In [ ]:
manifest_candidates = []
for path in INPUT_ROOT.rglob("manifest.json"):
    try:
        manifest = json.loads(path.read_text(encoding="utf-8"))
    except (OSError, UnicodeDecodeError, json.JSONDecodeError):
        continue
    if (
        manifest.get("dataset") == "REUTERS"
        and manifest.get("protocol_id") == SOURCE_PROTOCOL
        and manifest.get("training_commit") == SOURCE_TRAINING_COMMIT
        and manifest.get("seed") == 42
        and manifest.get("status") == "complete"
    ):
        manifest_candidates.append((path, manifest))
assert len(manifest_candidates) == 1, manifest_candidates
source_manifest, source_identity = manifest_candidates[0]

source_features = list(INPUT_ROOT.rglob("10k_feature.npy"))
assert len(source_features) == 1, source_features
source_targets = source_features[0].with_name("10k_target.npy")
assert source_targets.is_file(), source_targets

NORMALIZED_ROOT = WORKING_ROOT / "mvdec-reuters-audit-source"
if NORMALIZED_ROOT.exists():
    shutil.rmtree(NORMALIZED_ROOT)
run_dir = (
    NORMALIZED_ROOT
    / "runs"
    / source_identity["dataset"].lower()
    / source_identity["protocol_id"]
    / source_identity["config_hash"][:12]
    / "seed_42"
)
shutil.copytree(source_manifest.parent, run_dir)
MANIFEST = run_dir / "manifest.json"

DATASET_ROOT = NORMALIZED_ROOT / "datasets"
reuters_root = DATASET_ROOT / "REUTERS"
reuters_root.mkdir(parents=True)
shutil.copy2(source_features[0], reuters_root / source_features[0].name)
shutil.copy2(source_targets, reuters_root / source_targets.name)

print("source dataset", SOURCE_DATASET)
print("manifest", MANIFEST)
print("dataset root", DATASET_ROOT)


In [ ]:
if REPO.exists():
    shutil.rmtree(REPO)
subprocess.run(
    ["git", "clone", "https://github.com/vdat-18/mvdec_fpmax.git", str(REPO)],
    check=True,
)
subprocess.run(["git", "checkout", "--detach", AUDIT_COMMIT], cwd=REPO, check=True)
resolved = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=REPO, text=True
).strip()
assert resolved == AUDIT_COMMIT, (resolved, AUDIT_COMMIT)
subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "-q", "--no-deps",
        "gower", "kmodes", "loguru>=0.7.3", "mlxtend",
    ],
    check=True,
)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", str(REPO)],
    check=True,
)
print("audit commit", resolved)


In [ ]:
if OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)

environment = os.environ.copy()
command = [
    sys.executable,
    "-m",
    "pipeline.mvdec_paper_figure_audit",
    "--manifest",
    str(MANIFEST),
    "--dataset-root",
    str(DATASET_ROOT),
    "--output-dir",
    str(OUTPUT_DIR),
    "--inference-batch-size",
    "256",
    "--expected-seed",
    "42",
]
subprocess.run(
    command,
    cwd=REPO,
    env=environment,
    check=True,
    timeout=HARD_TIMEOUT_SECONDS,
)


In [ ]:
import pandas as pd

metrics_path = OUTPUT_DIR / "paper_figure_diagnostic_metrics.csv"
audit_manifest_path = OUTPUT_DIR / "paper_figure_diagnostic_manifest.json"
metrics = pd.read_csv(metrics_path)
audit_manifest = json.loads(audit_manifest_path.read_text(encoding="utf-8"))
expected_representations = {
    "eq4_latent_average",
    "figure1_full_output_average",
    "figure2_full_output_concatenation",
}
assert set(metrics["representation"]) == expected_representations
assert audit_manifest["claim_scope"] == "post_training_diagnostic_not_exact_paper_truth"
assert audit_manifest["source_run"]["dataset"] == "REUTERS"
assert audit_manifest["source_run"]["seed"] == 42
assert audit_manifest["runtime"]["numpy_version"] == EXPECTED_NUMPY
assert audit_manifest["runtime"]["scikit_learn_version"] == EXPECTED_SKLEARN

display(metrics)
print(json.dumps(audit_manifest, indent=2))


In [ ]:
archive_base = WORKING_ROOT / "mvdec-reuters-paper-figure-audit-seed42"
archive_path = Path(shutil.make_archive(str(archive_base), "zip", OUTPUT_DIR))
digest = hashlib.sha256(archive_path.read_bytes()).hexdigest()
sha_path = archive_path.with_suffix(archive_path.suffix + ".sha256")
sha_path.write_text(f"{digest}  {archive_path.name}\n", encoding="ascii")
print("archive", archive_path)
print("sha256", digest)
